In [0]:
%python
VOLUME_PATH = "/Volumes/supply_chain_demo/default/raw"
METADATA_PATH = "/Volumes/supply_chain_demo/default/raw/metadata/"

spark.sql(f"LIST '{VOLUME_PATH}'").display()
       
spark.sql(f"LIST '{METADATA_PATH}'").display()
       

In [0]:
%python
spark.sql(f"LIST '{VOLUME_PATH}/data'").display()

In [0]:
BASE_DIR = "/Volumes/supply_chain_demo/default/raw"

schema = (
    spark.read.format("csv")
    .options(header=True, inferSchema=True)
    .load(f"{BASE_DIR}/data/DataCoSupplyChainDataset.csv")
    .schema
)

schema

In [0]:
# Läs in bronze table
df = spark.sql("SELECT * FROM supply_chain_demo.bronze.raw_supply_chain")

display(df)

In [0]:

df = spark.sql("FROM supply_chain_demo.bronze.raw_supply_chain")
df.limit(5).display()

In [0]:
df.select("Customer Email", "Customer Country", "Benefit per order", "shipping date (DateOrders)",).limit(5).display()

In [0]:
import re 
from pyspark.sql.functions import to_timestamp,col, coalesce, lit, when, round, col, regexp_replace, date_format, trim
from pyspark.sql.types import DecimalType, IntegerType, DoubleType

def to_snake_case(name):
    return re.sub(r"[\s]+", "_", name.strip().casefold())

def rename_columns_to_snake_case(df):
    new_columns = [to_snake_case(column) for column in df.columns]
    return df.toDF(*new_columns)

df_column_alias = rename_columns_to_snake_case(df)

df_column_alias.limit(5).display()

In [0]:
df_timestamp = df_column_alias.withColumn(
    "shipping_date", to_timestamp(col("shipping_date_(dateorders)"), "M/d/yyyy H:mm")
)

df_timestamp.select("shipping_date", "shipping_date_(dateorders)").limit(2).display()

In [0]:
df_cleaned = (
    df_timestamp
    # Hantera Nulls 
    .withColumn("customer_lname", coalesce(col("customer_lname"), lit("-")))
    .withColumn("customer_zipcode", coalesce(col("customer_zipcode").cast("string"), lit("unknown")))
    .withColumn("order_zipcode", coalesce(col("order_zipcode").cast("string"), lit("unknown")))
    
    # Trimma strängar (Räddar mig från många buggar i framtiden!)
    .withColumn("customer_city", trim(col("customer_city")))
    .withColumn("order_city", trim(col("order_city")))
    .withColumn("order_status", trim(col("order_status")))
    
    # Datetime parsning
    .withColumn("shipping_date", to_timestamp(col("shipping_date_(dateorders)"), "M/d/yyyy H:mm"))
    .withColumn("order_date", to_timestamp(col("order_date_(dateorders)"), "M/d/yyyy H:mm"))
    
    # Standardisera länder 
    .withColumn(
        "customer_country",
        when(col("customer_country") == "EE. UU.", "United States")
        .otherwise(col("customer_country"))
    )
    
    # Fixa datatyper för Geografi och Flaggor
    .withColumn("latitude", col("latitude").cast(DoubleType()))
    .withColumn("longitude", col("longitude").cast(DoubleType()))
    .withColumn("late_delivery_risk", col("late_delivery_risk").cast(IntegerType()))
    .withColumn("product_status", col("product_status").cast(IntegerType()))

    # Casta finansiell data till Decimal(10,2) för exakt precision
    .withColumn("order_item_product_price", col("order_item_product_price").cast(DecimalType(10, 2)))
    .withColumn("product_price", col("product_price").cast(DecimalType(10, 2))) 
    .withColumn("order_item_total", col("order_item_total").cast(DecimalType(10, 2))) 
    .withColumn("benefit_per_order", col("benefit_per_order").cast(DecimalType(10, 2)))
    .withColumn("sales_per_customer", col("sales_per_customer").cast(DecimalType(10, 2)))
    .withColumn("sales", col("sales").cast(DecimalType(10, 2)))
    .withColumn("order_profit_per_order", col("order_profit_per_order").cast(DecimalType(10, 2)))
    
    # Casta ratios till Decimal(10,4)
    .withColumn("order_item_discount_rate", col("order_item_discount_rate").cast(DecimalType(10, 4)))
    .withColumn("order_item_profit_ratio", col("order_item_profit_ratio").cast(DecimalType(10, 4)))
    
    # Droppa enbart PII och de gamla datum-kolumnerna
).drop(
    "product_description", 
    "customer_email", 
    "customer_password", 
    "shipping_date_(dateorders)", 
    "order_date_(dateorders)",
    "product_image"
)    

df_cleaned.display()

In [0]:
df_cleaned.limit(4).display()
